In [1]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray          # registers the .rio accessor on xarray
from shapely.geometry import mapping
from pathlib import Path
from datetime import datetime
 
warnings.filterwarnings("ignore", category=FutureWarning)

## CONFIGURATION

edit these paths and variable names to match your files

In [36]:
# --- Paths ---
INPUT_DIR = Path("data")
SHAPEFILE_PATH = INPUT_DIR / "batas_provinsi/Provinsi_Kemdagri.shp"
 
YEARS = range(2021, 2026)  # 2021-2025
 
# TerraClimate variable name inside the .nc (usually matches the filename label)
NC_VARIABLES = {
    "tmax": "tmax",
    "tmin": "tmin",
    # "ppt":  "ppt",
}
 
OUTPUT_DIR      = Path("output")
CLIPPED_NC_DIR  = OUTPUT_DIR / "clipped_nc"
ZONAL_STATS_DIR = OUTPUT_DIR / "zonal_stats"
 
# Column in shapefile with province names
# Common: "NAME_1" (GADM), "PROVINSI", "ADM1_EN", "name"
PROVINCE_COL = "nama_prop"
 
TERRACLIMATE_CRS = "EPSG:4326"
CRS          = "EPSG:4326"
COMPUTE_ZONAL_STATS = True

# Set False to skip saving per-province .nc files (saves disk + time)
SAVE_CLIPPED_NC = True

 ## HELPER FUNCTIONS

In [13]:
def sanitize_name(name: str) -> str:
    return re.sub(r"[^a-zA-Z0-9]+", "_", name).strip("_").lower()

def open_nc(var, year):
    """Open one TerraClimate file, standardize dims, return DataArray."""
    f = INPUT_DIR / f"TerraClimate_{var}_{year}.nc"
    ds = xr.open_dataset(str(f), engine="netcdf4")
    da = ds[var]
 
    rename = {}
    if "lon" in da.dims: rename["lon"] = "x"
    if "lat" in da.dims: rename["lat"] = "y"
    if "longitude" in da.dims: rename["longitude"] = "x"
    if "latitude" in da.dims: rename["latitude"] = "y"
    if rename:
        da = da.rename(rename)
 
    da = da.rio.write_crs(CRS)
    da = da.rio.set_spatial_dims(x_dim="x", y_dim="y")
    if da.y[0] < da.y[-1]:
        da = da.sortby("y", ascending=False)
    return da, ds

def load_and_concat_yearly(variable: str, var_name: str) -> xr.DataArray:
    """
    Load TerraClimate_<var>_<year>.nc for each year, concat along time.
    """
    arrays = []
    for year in YEARS:
        fpath = INPUT_DIR / f"TerraClimate_{variable}_{year}.nc"
        if not fpath.exists():
            print(f"    WARNING: {fpath} not found, skipping")
            continue
        ds = xr.open_dataset(str(fpath), engine="netcdf4")
        da = ds[var_name]
        print(f"    {fpath.name}: time={da.sizes.get('time', '?')}")
        arrays.append(da)

    if not arrays:
        raise FileNotFoundError(f"No files found for '{variable}'")

    da = xr.concat(arrays, dim="time")

    # Standardize dim names for rioxarray
    rename = {}
    if "lon" in da.dims:
        rename["lon"] = "x"
    if "lat" in da.dims:
        rename["lat"] = "y"
    if "longitude" in da.dims:
        rename["longitude"] = "x"
    if "latitude" in da.dims:
        rename["latitude"] = "y"
    if rename:
        da = da.rename(rename)

    da = da.rio.write_crs(TERRACLIMATE_CRS)
    da = da.rio.set_spatial_dims(x_dim="x", y_dim="y")

    if da.y[0] < da.y[-1]:
        da = da.sortby("y", ascending=False)

    return da


def clip_to_polygon(da: xr.DataArray, geometry) -> xr.DataArray:
    return da.rio.clip([mapping(geometry)], crs=TERRACLIMATE_CRS,
                       drop=True, all_touched=True)


def compute_stats(clipped: xr.DataArray, province: str, var_label: str):
    records = []
    for t in range(clipped.sizes["time"]):
        s = clipped.isel(time=t)
        tv = pd.Timestamp(s.time.values)
        vals = s.values.flatten()
        vals = vals[~np.isnan(vals)]
        n = len(vals)
        records.append({
            "province": province,
            "variable": var_label,
            "year": tv.year,
            "month": tv.month,
            "date": tv.strftime("%Y-%m"),
            "mean":   float(np.mean(vals))   if n else np.nan,
            "median": float(np.median(vals)) if n else np.nan,
            "min":    float(np.min(vals))    if n else np.nan,
            "max":    float(np.max(vals))    if n else np.nan,
            "std":    float(np.std(vals))    if n else np.nan,
            "pixel_count": n,
        })
    return records

def zonal_stats(clipped, province, var):
    """Extract mean/min/max/std per timestep."""
    rows = []
    for t in range(clipped.sizes["time"]):
        s = clipped.isel(time=t)
        tv = pd.Timestamp(s.time.values)
        v = s.values.flatten()
        v = v[~np.isnan(v)]
        n = len(v)
        rows.append({
            "province": province, "variable": var,
            "year": tv.year, "month": tv.month,
            "date": tv.strftime("%Y-%m"),
            "mean":   float(np.mean(v))   if n else np.nan,
            "median": float(np.median(v)) if n else np.nan,
            "min":    float(np.min(v))    if n else np.nan,
            "max":    float(np.max(v))    if n else np.nan,
            "std":    float(np.std(v))    if n else np.nan,
            "pixel_count": n,
        })
    return rows

In [31]:
t0 = datetime.now()
print("=" * 70)
print("TerraClimate Batch Clip - Indonesia Provinces")
print(f"Started : {t0:%Y-%m-%d %H:%M:%S}")
print(f"Python  : {os.sys.version.split()[0]}")
print(f"NumPy   : {np.__version__}")
print(f"xarray  : {xr.__version__}")
print("=" * 70)

# --- Shapefile ---
gdf = gpd.read_file(SHAPEFILE_PATH)
print(f"\nShapefile: {len(gdf)} features")
print(f"Columns : {list(gdf.columns)}")

if PROVINCE_COL not in gdf.columns:
    print(f"\n*** '{PROVINCE_COL}' not found! Available: {list(gdf.columns)}")
    # return

if gdf.crs and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)


TerraClimate Batch Clip - Indonesia Provinces
Started : 2026-04-06 13:20:25
Python  : 3.8.20
NumPy   : 1.24.3
xarray  : 2023.1.0

Shapefile: 38 features
Columns : ['objectid', 'no_prop', 'nama_prop', 'jumlah_kab', 'jumlah_kot', 'jumlah_kec', 'jumlah_des', 'jumlah_kel', 'jumlah_pen', 'jumlah_kk', 'luas_wilay', 'kepadatan_', 'perpindaha', 'jml_mening', 'jml_lahir', 'perubahan_', 'jml_wktp', 'jml_rekam_', 'field1', 'field2', 'islam', 'kristen', 'katholik', 'hindu', 'budha', 'konghucu', 'kepercayaa', 'field3', 'field4', 'pria', 'wanita', 'field5', 'field6', 'belum_kawi', 'kawin', 'cerai_hidu', 'cerai_mati', 'field7', 'field8', 'u0', 'u5', 'u10', 'u15', 'u20', 'u25', 'u30', 'u35', 'u40', 'u45', 'u50', 'u55', 'u60', 'u65', 'u70', 'u75', 'field9', 'field10', 'lhr_2020', 'lhr_sebelu', 'lhr_2021', 'lhr_sebe_1', 'lhr_2022', 'lhr_sebe_2', 'lhr_2023', 'lhr_sebe_3', 'lhr_2024', 'lhr_sebe_4', 'pertumbuha', 'pertumbu_1', 'pertumbu_2', 'pertumbu_3', 'pertumbu_4', 'field11', 'field12', 'pendidikan', 'p

In [12]:
import gc
gc.collect()

1853

In [11]:
# --- Load & concat yearly NCs ---
data_arrays = {}
for label, var_name in NC_VARIABLES.items():
    print(f"\nLoading {label}:")
    da = load_and_concat_yearly(label, var_name)
    print(f"  Concatenated: {da.shape} ({da.sizes['time']} months)")
    data_arrays[label] = da

# --- Prep output ---
CLIPPED_NC_DIR.mkdir(parents=True, exist_ok=True)
ZONAL_STATS_DIR.mkdir(parents=True, exist_ok=True)



Loading ppt:
    TerraClimate_ppt_2021.nc: time=12
    TerraClimate_ppt_2022.nc: time=12
    TerraClimate_ppt_2023.nc: time=12
    TerraClimate_ppt_2024.nc: time=12
    TerraClimate_ppt_2025.nc: time=12


MemoryError: Unable to allocate 3.34 GiB for an array with shape (12, 4320, 8640) and data type float64

In [ ]:

all_stats = []
n = len(gdf)

# --- Clip loop ---
for i, (_, row) in enumerate(gdf.iterrows(), 1):
    prov = row[PROVINCE_COL]
    safe = sanitize_name(prov)
    pdir = CLIPPED_NC_DIR / safe
    pdir.mkdir(parents=True, exist_ok=True)

    print(f"\n[{i}/{n}] {prov}")

    for label, da in data_arrays.items():
        out = pdir / f"{label}_{safe}.nc"
        try:
            clipped = clip_to_polygon(da, row.geometry)

            clipped_ds = clipped.to_dataset(name=label)
            clipped_ds.attrs.update({
                "province": prov,
                "source": "TerraClimate",
                "period": "2021-2025",
            })
            clipped_ds.to_netcdf(out)

            kb = out.stat().st_size / 1024
            print(f"  {label:4s} -> {clipped.shape}  ({kb:.0f} KB)")

            if COMPUTE_ZONAL_STATS:
                all_stats.extend(compute_stats(clipped, prov, label))

        except Exception as e:
            print(f"  {label} FAILED: {e}")

# --- Zonal stats CSV ---
if COMPUTE_ZONAL_STATS and all_stats:
    df = pd.DataFrame(all_stats)
    sp = ZONAL_STATS_DIR / "zonal_summary.csv"
    df.to_csv(sp, index=False)
    print(f"\nZonal stats -> {sp}")
    print(f"  {len(df)} records | {df['province'].nunique()} provinces")

print(f"\nDone in {(datetime.now() - t0).total_seconds():.1f}s")

In [19]:
t0 = datetime.now()
print("=" * 60)
print("TerraClimate Batch Clip (low-memory mode)")
print("=" * 60)

# Load shapefile (small, stays in memory the whole time)
gdf = gpd.read_file(SHAPEFILE_PATH)
if PROVINCE_COL not in gdf.columns:
    print(f"Column '{PROVINCE_COL}' not found. Available: {list(gdf.columns)}")
    #return

if gdf.crs and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)
print(f"Provinces: {len(gdf)}")

TerraClimate Batch Clip (low-memory mode)
Provinces: 38


In [37]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
all_stats = []

# Outer loop: one file at a time (3 vars x 5 years = 15 iterations)
total_files = len(NC_VARIABLES) * len(YEARS)
file_i = 0

In [40]:
for var in NC_VARIABLES:
    for year in YEARS:
        file_i += 1
        fname = f"TerraClimate_{var}_{year}.nc"
        fpath = INPUT_DIR / fname

        if not fpath.exists():
            print(f"[{file_i}/{total_files}] {fname} — NOT FOUND, skip")
            continue

        print(f"\n[{file_i}/{total_files}] {fname}")
        da, ds = open_nc(var, year)

        # Clip each province against this single file
        for _, row in gdf.iterrows():
            prov = row[PROVINCE_COL]
            safe = sanitize_name(prov)

            try:
                clipped = da.rio.clip(
                    [mapping(row.geometry)], crs=CRS,
                    drop=True, all_touched=True
                )

                # Collect stats
                all_stats.extend(zonal_stats(clipped, prov, var))

                # Optionally save clipped .nc
                if SAVE_CLIPPED_NC:
                    pdir = OUTPUT_DIR / "clipped_nc" / safe
                    pdir.mkdir(parents=True, exist_ok=True)
                    out = pdir / f"{var}_{year}_{safe}.nc"
                    save_ds = clipped.to_dataset(name=var)
                    # Remove grid_mapping to prevent encoding conflict
                    for v in save_ds.data_vars:
                        save_ds[v].attrs.pop("grid_mapping", None)
                    save_ds.attrs.pop("grid_mapping", None)
                    save_ds.to_netcdf(out)

                del clipped

            except Exception as e:
                print(f"  {prov}: {e}")

        # Close and free memory before loading next file
        ds.close()
        del da, ds
        gc.collect()
        print(f"  done — {len(gdf)} provinces clipped")


[1/10] TerraClimate_tmax_2021.nc
  done — 38 provinces clipped

[2/10] TerraClimate_tmax_2022.nc
  done — 38 provinces clipped

[3/10] TerraClimate_tmax_2023.nc
  done — 38 provinces clipped

[4/10] TerraClimate_tmax_2024.nc
  done — 38 provinces clipped

[5/10] TerraClimate_tmax_2025.nc
  done — 38 provinces clipped

[6/10] TerraClimate_tmin_2021.nc
  done — 38 provinces clipped

[7/10] TerraClimate_tmin_2022.nc
  done — 38 provinces clipped

[8/10] TerraClimate_tmin_2023.nc
  done — 38 provinces clipped

[9/10] TerraClimate_tmin_2024.nc
  done — 38 provinces clipped

[10/10] TerraClimate_tmin_2025.nc
  done — 38 provinces clipped
